In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_path = "/content/drive/MyDrive/Data/processed_data/train_csv.csv"
valid_path = "/content/drive/MyDrive/Data/processed_data/valid_csv.csv"

In [ ]:
import pandas as pd
#read the csv from the training path and print first five rows
TrainData = pd.read_csv("/content/drive/MyDrive/Data/processed_data/train_csv.csv")
print(TrainData.head)

In [ ]:
ValidData = pd.read_csv(valid_path)
print(ValidData.head)

In [ ]:
train_label = TrainData["label"].unique()
train_label.sort()
print(train_label)

In [ ]:
valid_label = ValidData['label'].unique()
valid_label.sort()
print(valid_label)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize = (20,5))
sns.countplot(data = TrainData, x = "label")
plt.title("Label Distribution for Training Data")
plt.xlabel("Category")
plt.ylabel("Label count")
plt.show()

In [ ]:
plt.figure(figsize = (20,5))
sns.countplot(data = ValidData, x = "label")
plt.title("Label Distribution for Validation Data")
plt.xlabel("Category")
plt.ylabel("Label count")
plt.show()

In [ ]:
#label encoding (converting label categories into numerical values) for training data
TrainData['label']= pd.factorize(TrainData['label'], sort = True)[0]
print(TrainData.head)

In [ ]:
#label encoding for validation data
ValidData['label']= pd.factorize(ValidData['label'], sort = True)[0]
print(ValidData.head)


In [ ]:
#These are the features seperating into xTrain and xValid
xTrain = TrainData.iloc[:, :-1].values

xValid = ValidData.iloc[:, :-1].values

#Sperating the labels yTrain and yValid
yTrain = TrainData.iloc[:, -1].values

yValid = ValidData.iloc[:, -1].values

print(xTrain.shape)
print(yTrain.shape)
print(xValid.shape)
print(yValid.shape)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report




# Define function to evaluate the performance of model
def model_evaluations(y_true, y_pred):
  import matplotlib.pyplot as plt
  import seaborn as sns
  acc_score = accuracy_score(y_true, y_pred)
  print("Accuracy score: {}\n".format(acc_score))
  print("Classification Report: {}".format(classification_report(y_true, y_pred)))
  plt.figure(figsize = (10,10))
  sns.heatmap(confusion_matrix(y_true, y_pred),  annot = True, fmt="g", cmap = "Blues", xticklabels = train_label, yticklabels = train_label)
  plt.title("Consfuion Matrix")
  plt.show()

In [ ]:
#Build KNN algorithm
from sklearn.neighbors import KNeighborsClassifier
k = [i for i in range(2,16)]
acc = []
for k_value in k:
  main_model = KNeighborsClassifier(n_neighbors=k_value, n_jobs = -1)
  main_model.fit(xTrain, yTrain)
  ypred = main_model.predict(xValid)
  acctemp = accuracy_score(yValid, ypred)
  acc.append(acctemp)
  print(k_value, 'completed:', acctemp)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize = (10,10))
plt.plot(k, acc, color = "blue")
plt.xlabel("Number of Neighbors")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Number of Neighbors")
plt.grid(True)
plt.xticks([i for i in range(16)])
plt.show()


In [ ]:
#RandomForest

from sklearn.ensemble import RandomForestClassifier
max_depth = [1,2,3,4]
#[i for i in range(10,110,10)] list comprehension
number_of_trees = [50]
allacc = []
for depth in max_depth:
  acc = []
  for trees in number_of_trees:
    model = RandomForestClassifier(n_estimators=trees, max_depth= depth, n_jobs = -1)
    model.fit(xTrain, yTrain)
    ypred = model.predict(xValid)
    acctemp = accuracy_score(yValid, ypred)
    acc.append(acctemp)
    print(acctemp, "Depth: {} and Tree: {} done".format(depth, trees))
  allacc.append(acc)


In [ ]:
# MLP (Multi Layer Perceptron) Algorithm
from sklearn.neural_network import MLPClassifier
model1 = MLPClassifier(learning_rate_init=0.005, max_iter=20)
model1.fit(xTrain, yTrain)
ypred = model1.predict(xValid)
acc = accuracy_score(yValid, ypred)
print("Accuracy :", acc)

In [ ]:
!pip install lightgbm

In [ ]:
#lightGBM
from lightgbm import LGBMClassifier
depths = [8,9,10,11,12]
n_estimators_list = [10,20,30,40,50,60,70,80,90,100]
all_acc = []
for depth in depths:
  acc = []
  for trees in n_estimators_list:
    print("\n")
    print(f"Running LGBM")
    print("\n")
    model1 = LGBMClassifier(max_depth=depth,n_estimators=trees,learning_rate=0.001,verbosity=-1)
    model1.fit(xTrain,yTrain)
    y_pred = model1.predict(xValid)
    accuracy = accuracy_score(yValid, y_pred)
    acc.append(accuracy)
    print("Depth: {} and Tree: {} done: {}".format(depth, trees, accuracy))
  all_acc.append(acc)

In [ ]:
!pip install xgboost

In [ ]:
#XGboost
from xgboost import XGBClassifier
depths = [8,9,10,11,12]
n_estimators_list = [10,20,30,40,50,60,70,80,90,100]
all_acc = []
for depth in depths:
  acc = []
  for trees in n_estimators_list:
    print("\n")
    print(f"Running XGB")
    print("\n")
    model1 = XGBClassifier(max_depth=depth,n_estimators=trees,learning_rate=0.001)
    model1.fit(xTrain,yTrain)
    y_pred = model1.predict(xValid)
    accuracy = accuracy_score(yValid, y_pred)
    acc.append(accuracy)
    print("Depth: {} and Tree: {} done: {}".format(depth, trees, accuracy))
  all_acc.append(acc)

In [ ]:
#save the best model
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=50, max_depth= 12, n_jobs = -1)
model.fit(xTrain, yTrain)
ypred = model.predict(xValid)
acctemp = accuracy_score(yValid, ypred)
print(acctemp)




In [ ]:
import pickle
f = open("/content/drive/MyDrive/Model/RF_50trees_12depth","wb")
pickle.dump(model,f)
f.close()

In [ ]:
#get features
features = TrainData.drop('label', axis=1).columns

In [ ]:
#Top 10 important features
importances = model.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': features,'Gini Importance': importances})
feature_imp_df =feature_imp_df.sort_values(by='Gini Importance', ascending=False)
print(feature_imp_df.head(10))

In [ ]:
#Save top10 in a df
top_10_features = feature_imp_df.head(10)
top_10_features = top_10_features.sort_values(by='Gini Importance', ascending=False)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(30, 10))
#plt.barh(features, importances, color='skyblue') #for all features
plt.barh(top_10_features['Feature'], top_10_features['Gini Importance'], color='blue') #for top 10 features
plt.xlabel('Gini Importance')
plt.title('Feature Importance - Gini Importance')
plt.gca().invert_yaxis()
plt.show()